In [3]:
import json
import os
import uuid


class Product:

    def __init__(self, product_id, price, stock):
        self.id = product_id
        self.price = price
        self.stock = stock


class Cart:

    def __init__(self):
        self.items = {}

    def add(self, product_id, quantity):
        if quantity <= 0:
            raise ValueError("Invalid quantity")

        self.items[product_id] = self.items.get(product_id, 0) + quantity


class Order:

    def __init__(self, items, total):
        self.id = str(uuid.uuid4())
        self.items = items
        self.total = total


class Inventory:

    def __init__(self, file):
        self.file = file

        with open(file, "r") as f:
            data = json.load(f)

        self.products = {}

        for p in data:
            self.products[p["id"]] = Product(
                p["id"], p["price"], p["stock"]
            )

    def save(self):
        data = []

        for p in self.products.values():
            data.append({
                "id": p.id,
                "price": p.price,
                "stock": p.stock
            })

        with open(self.file, "w") as f:
            json.dump(data, f, indent=4)

    def place_order(self, cart, discount=0):

        if not 0 <= discount <= 100:
            raise ValueError("Invalid discount")

        total = 0

        # Validate all products first
        for product_id, quantity in cart.items.items():

            if product_id not in self.products:
                raise ValueError("Product not found")

            product = self.products[product_id]

            if quantity <= 0 or quantity > product.stock:
                raise ValueError("Invalid quantity or insufficient stock")

            total += product.price * quantity

        total = total * (1 - discount / 100)

        order = Order(cart.items.copy(), total)

        # Save old stock for rollback
        old_stock = {
            p.id: p.stock for p in self.products.values()
        }

        try:
            for product_id, quantity in cart.items.items():
                self.products[product_id].stock -= quantity

            self.save()

            if os.path.exists("orders.json"):
                with open("orders.json", "r") as f:
                    orders = json.load(f)
            else:
                orders = []

            orders.append({
                "id": order.id,
                "items": order.items,
                "total": order.total
            })

            with open("orders.json", "w") as f:
                json.dump(orders, f, indent=4)

        except Exception:
            for product_id, stock in old_stock.items():
                self.products[product_id].stock = stock

            self.save()
            raise

        return order


# Create catalog.json before running
inventory = Inventory("catalog.json")

cart = Cart()
cart.add("P1", 2)

order = inventory.place_order(cart, 10)

print("Order ID:", order.id)
print("Total:", order.total)
print("Remaining stock:", inventory.products["P1"].stock)

Order ID: 38180ea9-f321-45a1-9cc4-ad2b249b21f2
Total: 180.0
Remaining stock: 3
